### ЗАДАЧА: Доска задач поддержки

У команды поддержки есть входящий поток задач от клиентов.
Нужно собрать удобную модель, которая позволит быстро посмотреть:
- какие задачи ещё открыты,
- сколько часов планируется по каждому клиенту,
- у какого клиента сейчас самая большая нагрузка,
- как меняется состояние доски после закрытия одной из задач.

На входе у тебя есть несколько строк с данными о задачах.
На выходе должен получиться компактный отчёт по текущей доске.


In [3]:
# rows: ticket_id|client|title|estimate_hours|status
rows = [
    'TK-100|Acme|Ошибка в отчёте|3.5|new',
    'TK-101|Beta|Починить интеграцию|5|in_progress',
    'TK-102|Acme|Обновить доступы|2|new',
    'TK-103|Delta|Проверить выгрузку|1.5|closed',
]


class Ticket:
    allowed_statuses = {'new', 'in_progress', 'closed'}

    def __init__(self, ticket_id, client, title, estimate_hours, status):
        # TODO: сохранить ticket_id, client, title
        self.ticket_id = ticket_id
        self.client = client
        self.title = title

        # TODO: hours хранить через внутреннее поле self._estimate_hours
        self._estimate_hours = 0
        # TODO: значение estimate_hours пропустить через property/setter
        self.estimate_hours = estimate_hours

        # TODO: проверить, что status входит в allowed_statuses, иначе ValueError
        if status not in self.allowed_statuses:
            raise ValueError("Неккоректный статус")
        self._status = status

    @property
    def estimate_hours(self):
        # TODO: вернуть текущее число часов
        return self._estimate_hours

    @estimate_hours.setter
    def estimate_hours(self, value):
        # TODO: привести value к float
        value = float(value)
        # TODO: если value <= 0 -> raise ValueError('Hours must be > 0')
        # TODO: сохранить результат в self._estimate_hours
        if value <= 0:
            raise ValueError("Время должно быть > 0")
        self._estimate_hours = value

    def close(self):
        # TODO: перевести задачу в статус 'closed'
        self._status = 'closed'

    def reopen(self):
        # TODO: перевести задачу обратно в статус 'new'
        self._status = 'new'

    @classmethod
    def from_row(cls, row):
        # TODO: split по '|'
        parts = row.split("|")
        if len(parts) != 5:
            raise ValueError("Неккоректный формат строки")
        ticket_id, client, title, estimate_hours, status = parts
        return Ticket(ticket_id, client, title, estimate_hours, status)
        # TODO: ожидать 5 частей: ticket_id, client, title, estimate_hours, status
        # TODO: вернуть Ticket(...)

    def __repr__(self):
        # TODO: вернуть строку вида Ticket(ticket_id='...', client='...', status='...')
        return f"Ticket(ticket_id='{self.ticket_id}', client='{self.client}', status='{self._status}')"


class TicketBoard:
    def __init__(self):
        self.tickets = []

    def add(self, ticket):
        # TODO: добавить объект Ticket в self.tickets
        self.tickets.append(ticket)

    def load(self, rows):
        # TODO: для каждой строки создать Ticket.from_row(row)
        # TODO: добавить тикет в доску через add(...)
        for row in rows:
            ticket = Ticket.from_row(row)
            self.add(ticket)

    def open_tickets(self):
        # TODO: вернуть список тикетов, у которых status != 'closed'
        return [ticket for ticket in self.tickets if ticket._status != 'closed']

    def by_client(self, client):
        # TODO: вернуть список тикетов только нужного клиента
        return [ticket for ticket in self.tickets if ticket.client == client]

    def total_hours_by_client(self):
        # TODO: собрать dict вида client -> total_hours
        # TODO: суммировать estimate_hours по каждому клиенту
        hours_by_client = {}
        for ticket in self.tickets:
            if ticket.client not in hours_by_client:
                hours_by_client[ticket.client] = 0
            hours_by_client[ticket.client] += ticket.estimate_hours
        return hours_by_client

    def busiest_client(self):
        # TODO: использовать total_hours_by_client()
        # TODO: вернуть tuple (client, total_hours) с максимумом
        hours_by_client = self.total_hours_by_client()
        if not hours_by_client:
            return None
        busiest = max(hours_by_client.items(), key = lambda x: x[1])
        return busiest


board = TicketBoard()

# TODO: загрузить строки в board
board.load(rows)
# TODO: вывести все тикеты
print("Все тикеты: ")
for ticket in board.tickets:
    print(ticket)
# TODO: вывести только открытые тикеты
print("Открытые тикеты: ")
for ticket in board.open_tickets():
    print(ticket)
# TODO: вывести задачи клиента 'Acme'
print("Задачи клиента Acme:")
for ticket in board.by_client('Acme'):
    print(ticket)
# TODO: вывести total_hours_by_client()
print("Суммарные часы по клиентам", board.total_hours_by_client())
# TODO: вывести busiest_client()
print("Самый загруженный клиент: ", board.busiest_client())
# TODO: закрыть первую открытую задачу и снова вывести open_tickets()
if board.open_tickets():
    board.open_tickets()[0].close()

print("Открытые задачи после закрытия первой: ")
for ticket in board.open_tickets():
    print(ticket)


Все тикеты: 
Ticket(ticket_id='TK-100', client='Acme', status='new')
Ticket(ticket_id='TK-101', client='Beta', status='in_progress')
Ticket(ticket_id='TK-102', client='Acme', status='new')
Ticket(ticket_id='TK-103', client='Delta', status='closed')
Открытые тикеты: 
Ticket(ticket_id='TK-100', client='Acme', status='new')
Ticket(ticket_id='TK-101', client='Beta', status='in_progress')
Ticket(ticket_id='TK-102', client='Acme', status='new')
Задачи клиента Acme:
Ticket(ticket_id='TK-100', client='Acme', status='new')
Ticket(ticket_id='TK-102', client='Acme', status='new')
Суммарные часы по клиентам {'Acme': 5.5, 'Beta': 5.0, 'Delta': 1.5}
Самый загруженный клиент:  ('Acme', 5.5)
Открытые задачи после закрытия первой: 
Ticket(ticket_id='TK-101', client='Beta', status='in_progress')
Ticket(ticket_id='TK-102', client='Acme', status='new')
